 ### 1. a) Business Understanding
🎓 Backstory
An organization wants to improve its recruitment and talent acquisition process by leveraging Natural Language Processing (NLP) and Machine Learning. With an increasing number of job opportunities and applications across different industries, organizations receive job descriptions covering various roles and skill requirements.

Traditionally, job descriptions are classified manually based on job titles, responsibilities, and required skills. However, this approach can be time-consuming, inconsistent, and difficult to manage when dealing with large volumes of job postings.

The organization aims to develop an automated system that can understand the textual content of job descriptions and classify them into their respective job categories. By analyzing job descriptions and identifying meaningful patterns in the text, the system can automate the classification process and improve the efficiency of recruitment-related activities.
💼 Business Problem
The organization currently lacks an efficient, automated approach for classifying job descriptions into different job categories. This leads to:
- Difficulty manually categorizing large volumes of job descriptions.
- Time-consuming recruitment and job classification processes.
- Inconsistent classification due to manual interpretation of job descriptions.
- Challenges in identifying the appropriate job category based on required skills and responsibilities.
- Increased effort in organizing and managing job-related information.
These challenges affect recruitment efficiency, job organization, and the overall effectiveness of talent acquisition processes.
🎯 Business Objective
To analyze job description data and develop a machine learning model that automatically classifies job descriptions into their respective job categories using Natural Language Processing techniques.

The goal is to:
- Understand the textual patterns and important keywords associated with different job categories.
- Clean and preprocess job description text to improve data quality.
- Convert textual data into numerical features using TF-IDF vectorization.
- Train and evaluate multiple machine learning classification algorithms.
- Identify the best-performing model for job description classification.
- Develop an automated system that predicts job categories for new job descriptions.
This model will serve as an intelligent classification tool to automate job categorization, reduce manual effort, and improve the efficiency of managing job-related information.
⚠️ Constraints
1. Data Availability: The model's performance depends on the availability of sufficient and relevant job descriptions across different job categories. Limited examples for certain categories may affect classification performance.
2. Data Quality: Missing descriptions, duplicate records, inconsistent text formats, HTML tags, URLs, and irrelevant characters may affect the quality of the dataset and model performance.
3. Text Complexity: Job descriptions may contain technical terms, abbreviations, industry-specific keywords, and similar skills across multiple job categories, making classification challenging.
4. Feature Representation: Since machine learning algorithms require numerical inputs, textual descriptions must be converted into numerical representations using techniques such as TF-IDF. Important contextual information may not be fully captured through this approach.
5. Model Performance: Different classification algorithms may produce different results. Selecting an appropriate model requires evaluating their performance using suitable classification metrics.
6. Prediction Limitations: The model can only predict categories based on the patterns learned from the training dataset. Unusual job descriptions, unseen terminology, or descriptions belonging to categories not represented in the training data may result in incorrect predictions.

### DATA COLLECTION

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\user\Downloads\job_roles_combined_ready_10461.csv")

### DATA UNDERSTANDING

In [3]:
df.head()

,jobdescription,label
0,"SR. DEVELOPER - PERM - RICHMOND, VA OR CHARLOT...",Software Development
1,Hotwire Communications is a next-generation co...,Network Engineering
2,Greetings from Computer Enterprises IncLooking...,Software Development
3,-Architect who has past experience in WebCente...,Software Development
4,Mindbank Consulting Group currently has an urg...,Cybersecurity


In [4]:
df['label'].value_counts()

label
Software Development    8359
Network Engineering      796
Cybersecurity            630
Data Analytics           272
Data Science             254
UI/UX Design             150
Name: count, dtype: int64

In [5]:
df.shape

(10461, 2)

In [6]:
df.isna().sum()

jobdescription    0
label             0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df['text_length'] = df['jobdescription'].apply(len)
df.groupby('label')['text_length'].mean().sort_values(ascending=False)

label
Cybersecurity           2977.807937
Network Engineering     2370.346734
UI/UX Design            2285.500000
Data Science            2204.318898
Data Analytics          2147.988971
Software Development    1775.610719
Name: text_length, dtype: float64

### DATA CLEANING

In [9]:
df_cleaned = df.copy()

In [10]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [11]:
df_cleaned['jobdescription'] = df_cleaned['jobdescription'].apply(clean_text)
df_cleaned.head()

,jobdescription,label,text_length
0,sr developer perm richmond va or charlottesvil...,Software Development,2595
1,hotwire communications is a next generation co...,Network Engineering,1926
2,greetings from computer enterprises inclooking...,Software Development,1518
3,architect who has past experience in webcenter...,Software Development,463
4,mindbank consulting group currently has an urg...,Cybersecurity,6180


### NLP Preprocessing

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]

    return ' '.join(words)

In [13]:
df_cleaned['processed_text'] = df_cleaned['jobdescription'].apply(preprocess_text)

In [14]:
df_cleaned.head()

,jobdescription,label,text_length,processed_text
0,sr developer perm richmond va or charlottesvil...,Software Development,2595,sr developer perm richmond va charlottesville ...
1,hotwire communications is a next generation co...,Network Engineering,1926,hotwire communication next generation converge...
2,greetings from computer enterprises inclooking...,Software Development,1518,greeting computer enterprise inclooking oracle...
3,architect who has past experience in webcenter...,Software Development,463,architect past experience webcenter ap solutio...
4,mindbank consulting group currently has an urg...,Cybersecurity,6180,mindbank consulting group currently urgent imm...


### Train-Test Split

In [15]:
x = df_cleaned['processed_text']
y = df_cleaned['label']

In [16]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(
    x,y,test_size = 0.2,random_state = 42,stratify = y
)

In [17]:
print(x_train.shape)
print(x_test.shape)

print(y_train.shape)
print(y_test.shape)

(8368,)
(2093,)
(8368,)
(2093,)


### Feature Engineering

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

In [19]:
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)

print(x_train_tfidf.shape)
print(x_test_tfidf.shape)


(8368, 53367)
(2093, 53367)


In [20]:
print(x_train_tfidf[:5].toarray())

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


### Model Building

In [21]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

nb = MultinomialNB()
nb.fit(x_train_tfidf,y_train)
y_pred_nb = nb.predict(x_test_tfidf)

In [22]:
lr = LogisticRegression()
lr.fit(x_train_tfidf, y_train)
y_pred_lr = lr.predict(x_test_tfidf)

In [23]:
svm = LinearSVC(random_state=42)
svm.fit(x_train_tfidf, y_train)
y_pred_svm = svm.predict(x_test_tfidf)
y_pred_svm[:10]

array(['Software Development', 'Software Development',
       'Software Development', 'Software Development',
       'Software Development', 'Network Engineering',
       'Software Development', 'Software Development', 'Cybersecurity',
       'Software Development'], dtype=object)

### Model Evaluation

In [24]:
from sklearn.metrics import accuracy_score
models = {
    'Multinomial': y_pred_nb,
    'Logistic Regression': y_pred_lr,
    'Linear SVM': y_pred_svm
}

for name, predictions in models.items():
    accuracy = accuracy_score(y_test, predictions)
    print(name, ":", accuracy)

Multinomial : 0.8050645007166747
Logistic Regression : 0.9498327759197325
Linear SVM : 0.9617773530817009


In [25]:
new_job = """
We are looking for a data analyst with strong knowledge
of Python, SQL, Excel, Power BI and data visualization.
The candidate should perform data cleaning and exploratory
data analysis.
"""
cleaned_job = clean_text(new_job)

processed_job = preprocess_text(cleaned_job)

In [26]:
job_vector = vectorizer.transform([processed_job])

In [27]:
prediction = svm.predict(job_vector)
prediction

array(['Data Analytics'], dtype=object)

In [28]:
prediction = svm.predict(job_vector)

print("Predicted Job Category:", prediction[0])

Predicted Job Category: Data Analytics


### MODEL DEPLOYMENT

In [29]:
import warnings
warnings.filterwarnings('ignore')

In [30]:
import pickle

In [31]:
with open(r'tfidf_vectorizer.pkl','wb') as file:
    pickle.dump(vectorizer,file)

In [32]:
with open(r'svm_model.pkl','wb') as file:
    pickle.dump(svm,file)

In [37]:
import gradio as gr

In [35]:
def predict_job_category(job_description):

    if not job_description or not job_description.strip():
        return "Please enter a job description."

    cleaned_job = clean_text(job_description)
    processed_job = preprocess_text(cleaned_job)
    job_vector = vectorizer.transform([processed_job])
    prediction = svm.predict(job_vector)

    return prediction[0]

In [36]:
sample = """
We are looking for a data analyst with strong knowledge
of Python, SQL, Excel, Power BI and data visualization.
The candidate should perform data cleaning and analysis.
"""

predict_job_category(sample)

'Data Analytics'

In [38]:
demo = gr.Interface(
    fn=predict_job_category,

    inputs=gr.Textbox(
        lines=10,
        label="Enter Job Description",
        placeholder="Paste your job description here..."
    ),

    outputs=gr.Textbox(
        label="Predicted Job Category"
    ),

    title="Job Description Classification",
    description="An NLP and Machine Learning application that predicts job categories from job descriptions.",

    theme=gr.themes.Soft()
)

In [39]:
demo.launch(share = True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
